In [1]:
# ============================================
# NOTEBOOK 2: DATA CLEANING & NATION ANALYSIS
# Clean the dataset and analyse which African
# nations dominate European football
# ============================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load clean data
df = pd.read_csv('../data/processed/african_players_clean.csv')

print("Data loaded")
print(f"Shape: {df.shape}")
print(f"\nColumn types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())

Data loaded
Shape: (361, 12)

Column types:
player              str
nation_code         str
position            str
age               int64
club                str
league              str
birth_year        int64
goals             int64
assists           int64
matches_played    int64
minutes           int64
nationality         str
dtype: object

Missing values:
player             0
nation_code        0
position           0
age                0
club               0
league            36
birth_year         0
goals              0
assists            0
matches_played     0
minutes            0
nationality        0
dtype: int64


In [2]:
# ============================================
# CLEANING STEPS
# ============================================

# Step 1 — Fix missing league values
print("Before cleaning:")
print(f"Missing leagues: {df['league'].isnull().sum()}")

# Drop rows with no league — we can't analyse
# players we don't know the league for
df = df.dropna(subset=['league'])
print(f"\nAfter dropping missing leagues: {len(df)} players")

# Step 2 — Clean league names
league_map = {
    'ENG-Premier League': 'Premier League',
    'ESP-La Liga': 'La Liga',
    'ITA-Serie A': 'Serie A',
    'FRA-Ligue 1': 'Ligue 1',
    'GER-Bundesliga': 'Bundesliga'
}
df['league'] = df['league'].map(league_map).fillna(df['league'])

# Step 3 — Clean position codes
position_map = {
    'GK': 'Goalkeeper',
    'DF': 'Defender',
    'MF': 'Midfielder',
    'FW': 'Forward',
    'DF,MF': 'Defender/Midfielder',
    'MF,FW': 'Midfielder/Forward',
    'DF,FW': 'Defender/Forward'
}
df['position_clean'] = df['position'].map(position_map).fillna(df['position'])

# Step 4 — Add continent region
west_africa = ['Nigeria', 'Ghana', 'Senegal', 'Ivory Coast',
               'Mali', 'Guinea', 'Burkina Faso', 'Togo',
               'Benin', 'Sierra Leone', 'Liberia', 'Guinea-Bissau',
               'Gambia', 'Cape Verde']
north_africa = ['Morocco', 'Algeria', 'Tunisia', 'Egypt', 'Libya']
central_africa = ['Cameroon', 'DR Congo', 'Congo', 'Gabon',
                  'Equatorial Guinea']
east_africa = ['Ethiopia', 'Kenya', 'Uganda', 'Tanzania']
southern_africa = ['South Africa', 'Zimbabwe', 'Mozambique',
                   'Angola', 'Zambia']

def get_region(nationality):
    if nationality in west_africa: return 'West Africa'
    elif nationality in north_africa: return 'North Africa'
    elif nationality in central_africa: return 'Central Africa'
    elif nationality in east_africa: return 'East Africa'
    elif nationality in southern_africa: return 'Southern Africa'
    else: return 'Other Africa'

df['region'] = df['nationality'].apply(get_region)

# Step 5 — Convert numeric columns
for col in ['goals', 'assists', 'matches_played', 'minutes']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print("\nCleaning complete")
print(f"Final dataset: {df.shape}")
print(f"\nRegion distribution:")
print(df['region'].value_counts())
print(f"\nPosition distribution:")
print(df['position_clean'].value_counts())

Before cleaning:
Missing leagues: 36

After dropping missing leagues: 325 players

Cleaning complete
Final dataset: (325, 14)

Region distribution:
region
West Africa        197
North Africa        73
Central Africa      35
Southern Africa     11
Other Africa         9
Name: count, dtype: int64

Position distribution:
position_clean
Midfielder             101
Defender                75
Forward                 54
Midfielder/Forward      42
Defender/Midfielder     17
FW,MF                   16
MF,DF                   12
Goalkeeper               7
Defender/Forward         1
Name: count, dtype: int64


In [3]:
# ============================================
# ADD PERFORMANCE METRICS
# Goals per 90, assists per 90
# ============================================

# Calculate per 90 metrics
df['goals_per90'] = (df['goals'] / df['minutes'] * 90).round(3)
df['assists_per90'] = (df['assists'] / df['minutes'] * 90).round(3)
df['goal_contributions_per90'] = (
    (df['goals'] + df['assists']) / df['minutes'] * 90
).round(3)

# Replace infinity values with 0
df = df.replace([np.inf, -np.inf], 0)
df['goals_per90'] = df['goals_per90'].fillna(0)
df['assists_per90'] = df['assists_per90'].fillna(0)
df['goal_contributions_per90'] = df['goal_contributions_per90'].fillna(0)

# Save processed data
df.to_csv('../data/processed/african_players_clean.csv', index=False)
print("Processed data saved")
print(f"Final shape: {df.shape}")

# Top 10 performers by goal contributions per 90
print("\nTOP 10 AFRICAN PLAYERS BY GOAL CONTRIBUTIONS PER 90:")
print("(minimum 450 minutes played)")
top_performers = df[df['minutes'] >= 450].nlargest(
    10, 'goal_contributions_per90'
)[['player', 'nationality', 'club', 'league',
   'goals', 'assists', 'goal_contributions_per90']]
print(top_performers.to_string())

Processed data saved
Final shape: (325, 17)

TOP 10 AFRICAN PLAYERS BY GOAL CONTRIBUTIONS PER 90:
(minimum 450 minutes played)
              player  nationality            club          league  goals  assists  goal_contributions_per90
31     Mohamed Salah        Egypt       Liverpool  Premier League     29       18                     1.255
179     Amine Gouiri      Algeria       Marseille         Ligue 1     10        3                     1.116
180      Amine Harit      Morocco       Marseille         Ligue 1      2        4                     1.040
225     Sofiane Diop      Morocco            Nice         Ligue 1      6        4                     0.846
270  Ademola Lookman      Nigeria        Atalanta         Serie A     15        5                     0.801
3       Bryan Mbeumo     Cameroon       Brentford  Premier League     20        7                     0.712
5        Yoane Wissa     DR Congo       Brentford  Premier League     19        4                     0.709
220   Eva